# YOLOv8 Training Notebook (Ultralytics + Roboflow)

Notebook ini adalah template untuk:
- download dataset dari Roboflow
- training YOLOv8 dengan Ultralytics
- validasi dan inference hasil model

**Catatan:** ganti placeholder API key / workspace / project / version sesuai dataset kamu.

## 1) Install dependency
Jalankan cell ini di Google Colab atau Jupyter.

In [ ]:
!nvidia-smi

Tue Apr 14 04:27:12 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   39C    P8              9W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip install -q ultralytics roboflow
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))
import os
from ultralytics import YOLO
print('Ultralytics installed successfully')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 37.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.5/169.5 kB 20.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 20.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 83.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 107.3 MB/s eta 0:00:00
True
Tesla T4
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics installed successfully


## 2) Download dataset dari Roboflow
Paste snippet Roboflow kamu di bawah ini. Jika dataset sudah diekspor dalam format YOLOv8, pastikan `download('yolov8')`.

In [3]:
# ==============================
# ROBOFLOW DATASET DOWNLOAD
# ==============================
# Replace these placeholders with your Roboflow info.
# Example:
# from roboflow import Roboflow
# rf = Roboflow(api_key='YOUR_API_KEY')
# project = rf.workspace('YOUR_WORKSPACE').project('YOUR_PROJECT')
# version = project.version(YOUR_VERSION)
# dataset = version.download('yolov8')
# data_yaml = os.path.join(dataset.location, 'data.yaml')

from roboflow import Roboflow

ROBOFLOW_API_KEY = '1t0GmaYKwCkznFid2PRH'
WORKSPACE_NAME = 'roboflow-58fyf'
PROJECT_NAME = 'rock-paper-scissors-sxsw'
VERSION_NUMBER = 14

#bagian ini pake snip code dari roboflownya
rf = Roboflow(api_key="1t0GmaYKwCkznFid2PRH")
project = rf.workspace("roboflow-58fyf").project("rock-paper-scissors-sxsw")
version = project.version(14)
dataset = version.download("yolov8")


data_yaml = os.path.join(dataset.location, 'data.yaml')
print('Dataset location:', dataset.location)
print('data.yaml:', data_yaml)

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to rock-paper-scissors-14 in yolov8:: 100%|██████████| 14682/14682 [00:01<00:00, 9143.29it/s] 

Dataset location: /content/rock-paper-scissors-14
data.yaml: /content/rock-paper-scissors-14/data.yaml


## 3) Cek struktur dataset
Cell ini membantu memastikan path dataset sudah benar.

In [4]:
import os

print('Files in dataset folder:')
for root, dirs, files in os.walk(dataset.location):
    level = root.replace(dataset.location, '').count(os.sep)
    indent = '  ' * level
    print(f'{indent}{os.path.basename(root)}/')
    subindent = '  ' * (level + 1)
    for f in files[:10]:
        print(f'{subindent}{f}')

Files in dataset folder:
rock-paper-scissors-14/
  data.yaml
  README.dataset.txt
  README.roboflow.txt
  train/
    images/
      piscina_cubierta_07_10_altavista_jpg.rf.1206c92560224ccc15b5cf74b5cf6d62.jpg
      youtube-38_jpg.rf.0521a412ff34da89608215a911bed226.jpg
      IMG_7079_MOV-3_jpg.rf.6fd14a4ae67eaf61cdc7814e4c00226b.jpg
      egohands-public-1624298525239_png_jpg.rf.5c3d8d2b749fb68e0a65d725a8fd9f56.jpg
      Screen-Recording-2023-03-11-at-9_48_18-PM_mov-145_jpg.rf.f118edcd4938adc09b991c22eb0bb0a5.jpg
      IMG_5567_mp4-12_jpg.rf.8a0b513f0c56799ae03f0de180eb4fe8.jpg
      egohands-public-1624576225725_png_jpg.rf.dbdc68c869f8d6403139b4c4914f299e.jpg
      egohands-public-1623358712656_png_jpg.rf.5fa9af70059b7b6b6a2c96f759e0b87a.jpg
      egohands-public-1625679154454_png_jpg.rf.28744d840dae993f7fe24dec74c0db40.jpg
      egohands-public-1623941905051_png_jpg.rf.b1a565cbef9bd4650127cb139903fa53.jpg
    labels/
      IMG_7077_MOV-70_jpg.rf.4edafe21320bc7e82dccd09bf108eeca.txt
  

## 4) Load model YOLOv8
Pakai model kecil dulu untuk training awal.

In [5]:
# Choose one:
# yolov8n.pt = nano, paling ringan
# yolov8s.pt = small, masih relatif ringan
# yolov8m.pt = medium

model = YOLO('yolov8s.pt')
print('Model loaded:', model.model_name if hasattr(model, 'model_name') else 'YOLOv8n')

Model loaded: yolov8s.pt


## 5) Training
Atur epoch, image size, batch size, dan device sesuai resource.

In [6]:
results = model.train(
    data=data_yaml,
    epochs=7, #ini boleh diubah2 sampe kamu temu yg cocok untuk datasetnhya
    imgsz=640,
    batch=64, #ini bisa di ubah2, kalau pake T4 mungkin bisa ampe 64, tergantung si, coba2 ja
    patience=20,
    device=0,   # ganti 'cpu' jika tidak ada GPU
    project='runs/train',
    name='yolov8_roboflow'
)

print(results)

Ultralytics 8.4.37 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=64, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/rock-paper-scissors-14/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=7, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolov8_roboflow, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=T

## 6) Validation
Evaluasi model setelah training selesai.

In [7]:
metrics = model.val(data=data_yaml)
print(metrics)

Ultralytics 8.4.37 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 11,126,745 parameters, 0 gradients, 28.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1484.1±464.0 MB/s, size: 29.5 KB)
val: Scanning /content/rock-paper-scissors-14/valid/labels.cache... 576 images, 238 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 576/576 241.6Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 36/36 3.9it/s 9.3s
                   all        576        400       0.91      0.905      0.933      0.696
                 Paper        132        139      0.923      0.868      0.927      0.678
                  Rock        121        141      0.934      0.929      0.942      0.714
              Scissors        116        120      0.872      0.917      0.928      0.696
Speed: 2.0ms preprocess, 10.7ms inference, 0.0ms loss, 0.9ms postprocess per image
Results saved to /content/runs/dete

 Tips praktis
- Mulai dari `yolov8n.pt` dulu.
- Pastikan dataset Roboflow sudah dalam format YOLOv8.
- Kalau GPU Colab terbatas, kecilkan batch size.
- Setelah training, cek `runs/train/.../weights/best.pt`.


##Praktikum

1. cari dataset dari roboflow (boleh buat, boleh ngambil)
2. ambil sniped codenya dan lakukan training disini
3. cari hasil best.pt pada /content/runs/detect/runs/train/yolov8_roboflow/weights/best.pt
4. cari hasil graph, matrix, kurva, dll, yg berupa hasil dari training. (terletak di /content/runs/detect/runs/train/yolov8_roboflow)


#Tugas
1. cari hasil best.pt
2. lakukan inferensi dengan model baru hasil training. gunakan python dan openCV. (ada bounding box dan label classnya terbaca)
3. Lakukan pendalaman mengenai hasil2 graph, curve, dan matrix. cari tahu cara bacanya yaaa!
4.  pada bagian 5) training, lakukan pendalaman mengenai parameternya (epoch, batch, dll) apa pengaruhnya, dia ngapain di training. cari tahu sajaa, otak atik juga boleee siii.

#tips
1. kalau cari/buat dataset yg kecil aja, jangan banyak2. tp bebas sih
2. epoch ga perlu besar2, lama soalnya :(
3. pake google colab GPU T4 yaa



refrensi:

https://docs.ultralytics.com/modes/train/

https://youtu.be/a3SBRtILjPI?si=GcsQ___h7Xo9Vp6S

https://youtu.be/r0RspiLG260?si=f2UzNq5CTQ0dIaLF

